In [ ]:
"""
DINOv3 fine-tuning for ai-of-god-5 (72 animal classes, colourised segmentation overlays, macro-F1).

What is in here
  * DINOv3 ViT backbone (timm) + multi-layer feature fusion (CLS + mean-patch of the last N blocks, as in the
    DINOv2/DINOv3 linear-probe recipe) -> LayerNorm -> MLP head (same head style as your DINOv2 run: 512 hidden).
  * Loss = cross-entropy (label smoothing) on both views + lambda * SupCon on a projection of the fused feature.
    Two differently-augmented views of every image; the SupCon term pulls views and same-class images together,
    which explicitly teaches colour/appearance invariance.
  * Class-balanced PK batches (P classes x K images) - good for macro-F1 and gives SupCon real positives.
  * Layer-wise LR decay, head-only warm-up (frozen backbone), cosine schedule, EMA, fp16 AMP, early stopping.
  * Optional pseudo-label stage using confident TEST predictions from any ensemble (cfg['pseudo_npz']).
  * Saves: best/last .pt (with metadata), val+test probabilities .npz (same format as the other scripts, so
    build_ensemble_v2.py picks it up), submission csv, history json.

Needs a recent timm (pip install -U timm) and either internet (HF hub, DINOv3 licence) or cfg['pretrained_file'].
Run:  %run train_dinov3.py
"""
import os, re, json, math, random, time
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import timm

CFG = dict(
    comp_dir        = '/kaggle/input/competitions/ai-of-god-5/participant_data',
    out_dir         = '/kaggle/working',
    model_name      = 'vit_base_patch16_dinov3.lvd1689m',   # or vit_small_patch16_dinov3.lvd1689m (faster)
    pretrained_file = None,            # local state_dict for the backbone if internet is off
    tag             = 'dinov3_b',
    img_size        = 224,             # multiple of 16 (patch 16); 256/288 also fine (RoPE), but slower
    max_side        = 768,
    epochs          = 30,
    freeze_epochs   = 3,               # head-only warm-up (backbone frozen)
    P               = 16, K = 2,       # batch = P classes x K images (x2 views) -> 32 images, 64 forward passes
    lr              = 1e-4,
    head_lr_factor  = 10.0,
    layer_decay     = 0.75,
    weight_decay    = 0.05,
    drop_path       = 0.1,
    dropout         = 0.2,
    hidden_dim      = 512,
    n_last          = 4,               # fuse the last N blocks
    label_smoothing = 0.1,
    supcon_weight   = 0.5,             # set 0 to disable the contrastive term (ablation)
    supcon_temp     = 0.1,
    proj_dim        = 128,
    grad_clip       = 1.0,
    ema_decay       = 0.998,
    aug             = 'medium',        # 'mild' | 'medium' | 'strong'  (strong = recipe that did badly on the LB)
    patience        = 8,
    num_workers     = 4,
    seed            = 42,
    val_split       = 0.15,
    stratify        = True,            # auto-overridden if a reference val_ids file is found
    ref_states      = ['/kaggle/working/ensemble_state_v1.npz', '/kaggle/working/eva02_b_probs.npz'],
    full_train      = False,           # True: train on all labelled data, fixed epochs, save EMA each epoch
    pseudo_npz      = None,            # npz with test_ids + test_probs (ensemble probs) -> pseudo-label stage
    pseudo_thr      = 0.90,
    pseudo_cap      = 30,              # max pseudo-labelled images per class
)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ------------------------------------------------------------------ data
def get_split(train, cfg):
    def split(strat):
        return train_test_split(train, test_size=cfg['val_split'], random_state=cfg['seed'],
                                stratify=train.animal_class if strat else None)
    for ref_path in cfg['ref_states']:
        if os.path.exists(ref_path):
            ref = set(str(i) for i in np.load(ref_path, allow_pickle=True)['val_ids'])
            for strat in (True, False):
                tr, va = split(strat)
                if set(str(i) for i in va.image_id) == ref:
                    print(f'Split matches {os.path.basename(ref_path)} (stratify={strat})'); return tr, va
            print(f'WARNING: no split variant matches {ref_path}')
    return split(cfg['stratify'])


class RandomChannelPermute:
    def __init__(self, p): self.p = p
    def __call__(self, img):
        if random.random() < self.p:
            ch = img.split(); perm = random.sample(range(3), 3)
            img = Image.merge('RGB', [ch[i] for i in perm])
        return img


class RandomColorMatrix:
    def __init__(self, p, alpha=0.5): self.p, self.alpha = p, alpha
    def __call__(self, x):
        if random.random() < self.p:
            M = torch.distributions.Dirichlet(torch.full((3, 3), self.alpha)).sample()
            x = torch.einsum('ij,jhw->ihw', M, x)
        return x


AUG = dict(
    mild=  dict(cp=0.3, hue=0.15, sat=0.3, gray=0.05, blur=0.0,  erase=0.0,  cm=0.0, scale=0.7),
    medium=dict(cp=0.5, hue=0.30, sat=0.4, gray=0.10, blur=0.1,  erase=0.15, cm=0.0, scale=0.6),
    strong=dict(cp=0.7, hue=0.50, sat=0.5, gray=0.20, blur=0.15, erase=0.25, cm=0.4, scale=0.5),
)

def build_transforms(size, mean, std, level):
    L = AUG[level]
    ops = [T.RandomResizedCrop(size, scale=(L['scale'], 1.0), ratio=(0.75, 1.33),
                               interpolation=T.InterpolationMode.BICUBIC),
           T.RandomHorizontalFlip(),
           RandomChannelPermute(L['cp']),
           T.RandomApply([T.ColorJitter(0.3, 0.3, L['sat'], L['hue'])], p=0.8),
           T.RandomGrayscale(L['gray'])]
    if L['blur']: ops.append(T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=L['blur']))
    ops.append(T.ToTensor())
    if L['cm']: ops.append(RandomColorMatrix(L['cm']))
    ops.append(T.Normalize(mean, std))
    if L['erase']: ops.append(T.RandomErasing(p=L['erase'], scale=(0.02, 0.12), value='random'))
    eval_tf = T.Compose([T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC),
                         T.ToTensor(), T.Normalize(mean, std)])
    return T.Compose(ops), eval_tf


def load_img(path, max_side):
    img = Image.open(path).convert('RGB')
    if max_side and max(img.size) > max_side: img.thumbnail((max_side, max_side), Image.BILINEAR)
    return img


class TrainDS(Dataset):
    """Returns two independently augmented views of the same image."""
    def __init__(self, paths, labels, tf, max_side): self.paths, self.labels, self.tf, self.ms = paths, labels, tf, max_side
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = load_img(self.paths[i], self.ms)
        return self.tf(img), self.tf(img), int(self.labels[i])


class EvalDS(Dataset):
    def __init__(self, paths, tf, max_side): self.paths, self.tf, self.ms = paths, tf, max_side
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(load_img(self.paths[i], self.ms))


class PKBatchSampler:
    """Each batch: P random classes x K random images of each (class-balanced)."""
    def __init__(self, labels, P, K, seed):
        labels = np.asarray(labels)
        self.by = {c: np.where(labels == c)[0] for c in np.unique(labels)}
        self.cls = list(self.by); self.P, self.K, self.seed, self.n, self.epoch = P, K, seed, len(labels), 0
    def __len__(self): return self.n // (self.P * self.K)
    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch); self.epoch += 1
        for _ in range(len(self)):
            cs = rng.choice(len(self.cls), self.P, replace=len(self.cls) < self.P)
            batch = []
            for c in cs:
                idx = self.by[self.cls[c]]
                batch += rng.choice(idx, self.K, replace=len(idx) < self.K).tolist()
            yield batch


def load_pseudo(cfg, n_classes, comp):
    if not cfg['pseudo_npz']: return [], np.array([], dtype=int)
    Z = np.load(cfg['pseudo_npz'], allow_pickle=True)
    ids, P = [str(i) for i in Z['test_ids']], np.asarray(Z['test_probs'])
    conf, lab = P.max(1), P.argmax(1); keep = []
    for c in range(n_classes):
        idx = np.where((lab == c) & (conf >= cfg['pseudo_thr']))[0]
        keep += idx[np.argsort(-conf[idx])][:cfg['pseudo_cap']].tolist()
    print(f'pseudo-labels: kept {len(keep)}/{len(ids)} test images (thr {cfg["pseudo_thr"]}, cap {cfg["pseudo_cap"]}/class), '
          f'{len(set(lab[keep].tolist()))} classes covered')
    return [f'{comp}/test_images/{ids[i]}' for i in keep], lab[keep]


# ------------------------------------------------------------------ model
class Net(nn.Module):
    def __init__(self, cfg, n_classes, pretrained):
        super().__init__()
        kw = dict(pretrained=pretrained and cfg['pretrained_file'] is None, num_classes=0,
                  drop_path_rate=cfg['drop_path'])
        try:
            self.backbone = timm.create_model(cfg['model_name'], img_size=cfg['img_size'], **kw)
        except Exception as e:
            print('create_model failed:', e); print('available:', timm.list_models('*dinov3*'))
            raise
        if pretrained and cfg['pretrained_file']:
            pf = cfg['pretrained_file']
            if pf.endswith('.safetensors'):
                from safetensors.torch import load_file
                sd = load_file(pf)
            else:
                sd = torch.load(pf, map_location='cpu')
            sd = sd.get('state_dict', sd.get('model', sd)); msd = self.backbone.state_dict()
            sd = {k: v for k, v in sd.items() if k in msd and v.shape == msd[k].shape}
            if len(sd) < 0.8 * len(msd):      # wrong key format would otherwise load almost nothing, silently
                raise RuntimeError(f'only {len(sd)}/{len(msd)} backbone tensors matched. pretrained_file must be the '
                                   'timm-format weights (model.safetensors from the timm/<model_name> repo), '
                                   'not the transformers-format facebook/dinov3-* files.')
            print('loaded local pretrained backbone:', self.backbone.load_state_dict(sd, strict=False))
        self.n_last, self.npfx = cfg['n_last'], self.backbone.num_prefix_tokens
        self._feats = []
        for blk in self.backbone.blocks[-self.n_last:]:
            blk.register_forward_hook(self._hook)
        fd = 2 * self.n_last * self.backbone.num_features
        self.norm = nn.LayerNorm(fd)
        self.head = nn.Sequential(nn.Dropout(cfg['dropout']), nn.Linear(fd, cfg['hidden_dim']), nn.GELU(),
                                  nn.Dropout(cfg['dropout']), nn.Linear(cfg['hidden_dim'], n_classes))
        self.proj = nn.Sequential(nn.Linear(fd, cfg['hidden_dim']), nn.GELU(), nn.Linear(cfg['hidden_dim'], cfg['proj_dim']))

    def _hook(self, module, inp, out):
        self._feats.append(out[0] if isinstance(out, (tuple, list)) else out)

    def features(self, x):
        self._feats.clear()
        self.backbone.forward_features(x)                 # runs all blocks; hooks collect the last N
        outs, fs = list(self._feats), []
        self._feats.clear()
        for o in outs:
            o = self.backbone.norm(o)
            fs += [o[:, 0], o[:, self.npfx:].mean(1)]     # CLS + mean patch token
        return self.norm(torch.cat(fs, 1))

    def forward(self, x, with_proj=False):
        f = self.features(x)
        return (self.head(f), self.proj(f)) if with_proj else self.head(f)


def supcon_loss(z, labels, temp):
    """z: (n, d) L2-normalised; positives = every other sample with the same label (incl. the other view)."""
    n = z.size(0); eye = torch.eye(n, dtype=torch.bool, device=z.device)
    sim = (z @ z.T / temp).masked_fill(eye, -1e9)
    logp = sim - torch.logsumexp(sim, 1, keepdim=True)
    pos = (labels[:, None] == labels[None, :]) & ~eye
    return (-(logp * pos).sum(1) / pos.sum(1).clamp(min=1)).mean()


def param_groups(net, lr_unused, wd, decay, head_factor):
    L = len(net.backbone.blocks); groups = {}
    for n, p in net.named_parameters():
        if n.startswith('backbone.'):
            m = re.match(r'backbone\.blocks\.(\d+)\.', n)
            if m: lid = int(m.group(1)) + 1
            elif any(k in n for k in ('cls_token', 'reg_token', 'storage', 'pos_embed', 'patch_embed', 'rope', 'mask_token')): lid = 0
            else: lid = L + 1
            scale, is_head = decay ** (L + 1 - lid), False
        else:
            scale, is_head = head_factor, True
        no_wd = p.ndim <= 1 or n.endswith('.bias') or any(k in n for k in ('cls_token', 'reg_token', 'pos_embed', 'storage'))
        groups.setdefault((scale, no_wd, is_head), []).append(p)
    return [dict(params=ps, lr_scale=s, is_head=h, weight_decay=0.0 if nw else wd) for (s, nw, h), ps in groups.items()]


class EMA:
    def __init__(self, net, cfg, n_classes, decay):
        self.module = Net(cfg, n_classes, pretrained=False).to(dev)     # rebuilt (not deepcopy) so hooks bind correctly
        self.module.load_state_dict(net.state_dict()); self.module.eval(); self.decay = decay
        for p in self.module.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, net, step):
        d = min(self.decay, (1 + step) / (10 + step)); msd = net.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point: v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else: v.copy_(msd[k])


def apply_view(x, view, mean, std):
    """Apply TTA to normalized images. Channel permutations happen in RGB space,
    then the image is normalized again using the model's original mean/std."""
    if view == 'id':
        return x
    if view == 'flip':
        return x.flip(3)
    if view in ('p1', 'p2'):
        mu = torch.as_tensor(mean, device=x.device, dtype=x.dtype).view(1, 3, 1, 1)
        sigma = torch.as_tensor(std, device=x.device, dtype=x.dtype).view(1, 3, 1, 1)
        rgb = x * sigma + mu
        order = [1, 2, 0] if view == 'p1' else [2, 0, 1]
        rgb = rgb[:, order, :, :]
        return (rgb - mu) / sigma
    raise ValueError(f'Unknown TTA view: {view}')

@torch.no_grad()
def predict_views(net, loader, views=('id',), mean=(0.485, 0.456, 0.406),
                  std=(0.229, 0.224, 0.225)):
    net.eval(); out = {v: [] for v in views}
    for x in loader:
        x = x.to(dev, non_blocking=True)
        for v in views:
            xv = apply_view(x, v, mean, std)
            with torch.autocast(dev, dtype=torch.float16, enabled=dev == 'cuda'):
                out[v].append(net(xv).float().softmax(1).cpu())
    return {v: torch.cat(o).numpy() for v, o in out.items()}


def load_for_inference(path):
    """Later:  net, ck = load_for_inference('/kaggle/working/dinov3_b_best.pt')"""
    ck = torch.load(path, map_location='cpu', weights_only=False)
    net = Net(ck['cfg'], len(ck['classes']), pretrained=False)
    net.load_state_dict(ck['state_dict']); return net.to(dev).eval(), ck


# ------------------------------------------------------------------ main
def main(cfg=CFG):
    set_seed(cfg['seed']); comp, out, tag = cfg['comp_dir'], cfg['out_dir'], cfg['tag']
    print(f'Device: {dev}' + (f' ({torch.cuda.get_device_name(0)})' if dev == 'cuda' else ' — WARNING: CPU training will be very slow'), flush=True)
    for required in ('train.csv', 'sample_submission.csv', 'train_images', 'test_images'):
        path = f'{comp}/{required}'
        if not os.path.exists(path):
            raise FileNotFoundError(f"Required competition input not found: {path}. Set CFG['comp_dir'] to the participant_data folder.")
    train = pd.read_csv(f'{comp}/train.csv')
    classes = sorted(train.animal_class.unique()); assert len(classes) == 72
    cmap = {c: i for i, c in enumerate(classes)}
    tr, va = get_split(train, cfg)
    if cfg['full_train']: tr = train
    img_dir = f'{comp}/train_images'
    tr_paths = [f'{img_dir}/{i}' for i in tr.image_id]; tr_lab = tr.animal_class.map(cmap).values
    ps_paths, ps_lab = load_pseudo(cfg, 72, comp)
    tr_paths, tr_lab = tr_paths + ps_paths, np.concatenate([tr_lab, ps_lab]).astype(int)
    yva = va.animal_class.map(cmap).values
    print(f'{cfg["model_name"]}  train {len(tr_paths)} (pseudo {len(ps_paths)})  val {len(va)}  aug={cfg["aug"]}')

    net = Net(cfg, 72, pretrained=True).to(dev)
    pc = net.backbone.pretrained_cfg; mean, std = pc['mean'], pc['std']
    train_tf, eval_tf = build_transforms(cfg['img_size'], mean, std, cfg['aug'])
    kw = dict(num_workers=cfg['num_workers'], pin_memory=True, persistent_workers=cfg['num_workers'] > 0)
    sampler = PKBatchSampler(tr_lab, cfg['P'], cfg['K'], cfg['seed'])
    dl_tr = DataLoader(TrainDS(tr_paths, tr_lab, train_tf, cfg['max_side']), batch_sampler=sampler, **kw)
    dl_va = DataLoader(EvalDS([f'{img_dir}/{i}' for i in va.image_id], eval_tf, cfg['max_side']), batch_size=64, **kw)

    opt = torch.optim.AdamW(param_groups(net, cfg['lr'], cfg['weight_decay'], cfg['layer_decay'], cfg['head_lr_factor']),
                            lr=cfg['lr'], betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler('cuda', enabled=dev == 'cuda')
    ema = EMA(net, cfg, 72, cfg['ema_decay'])
    spe = len(sampler); freeze_steps = cfg['freeze_epochs'] * spe; total = cfg['epochs'] * spe
    def lr_at(step):
        if step < freeze_steps: return cfg['lr']
        s = step - freeze_steps
        return 1e-6 + 0.5 * (cfg['lr'] - 1e-6) * (1 + math.cos(math.pi * s / max(1, total - freeze_steps)))

    meta = dict(classes=classes, mean=mean, std=std, cfg=cfg)
    best_f1, best_ep, bad, step, history = -1, -1, 0, 0, []
    best_path, last_path = f'{out}/{tag}_best.pt', f'{out}/{tag}_last.pt'
    net.backbone.requires_grad_(False)                                  # head-only warm-up

    for ep in range(cfg['epochs']):
        if ep == cfg['freeze_epochs']: net.backbone.requires_grad_(True); print('backbone unfrozen')
        net.train()
        if ep < cfg['freeze_epochs']: net.backbone.eval()
        t0, run_ce, run_sc = time.time(), 0.0, 0.0
        for batch_i, (x1, x2, y) in enumerate(dl_tr, start=1):
            x = torch.cat([x1, x2]).to(dev, non_blocking=True); y = torch.cat([y, y]).to(dev)
            ramp = 1.0 if step < freeze_steps else min(1.0, (step - freeze_steps + 1) / spe)
            for g in opt.param_groups: g['lr'] = lr_at(step) * g['lr_scale'] * (1.0 if g['is_head'] else ramp)
            with torch.autocast(dev, dtype=torch.float16, enabled=dev == 'cuda'):
                logits, z = net(x, with_proj=True)
            ce = F.cross_entropy(logits.float(), y, label_smoothing=cfg['label_smoothing'])
            sc = supcon_loss(F.normalize(z.float(), dim=1), y, cfg['supcon_temp']) if cfg['supcon_weight'] > 0 else ce * 0
            loss = ce + cfg['supcon_weight'] * sc
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(net.parameters(), cfg['grad_clip'])
            scaler.step(opt); scaler.update(); ema.update(net, step)
            run_ce += ce.item(); run_sc += float(sc); step += 1
            if batch_i == 1 or batch_i % 10 == 0:
                print(f'ep {ep+1:02d}/{cfg["epochs"]} batch {batch_i:03d}/{spe} '
                      f'CE {ce.item():.3f} SupCon {float(sc):.3f} elapsed {time.time()-t0:.0f}s', flush=True)
        print('Training batches finished; running raw and EMA validation...', flush=True)
        f1_raw = f1_score(yva, predict_views(net, dl_va, mean=mean, std=std)['id'].argmax(1), average='macro')
        f1_ema = f1_score(yva, predict_views(ema.module, dl_va, mean=mean, std=std)['id'].argmax(1), average='macro')
        use_ema = f1_ema >= f1_raw or cfg['full_train']; f1 = f1_ema if use_ema else f1_raw
        sd = ema.module.state_dict() if use_ema else net.state_dict()
        history.append(dict(epoch=ep + 1, ce=run_ce / spe, supcon=run_sc / spe, f1_raw=f1_raw, f1_ema=f1_ema))
        print(f'ep {ep+1:02d}/{cfg["epochs"]}  ce {run_ce/spe:.3f}  supcon {run_sc/spe:.3f}  '
              f'val F1 raw {f1_raw:.4f} ema {f1_ema:.4f}  {time.time()-t0:.0f}s')
        torch.save(dict(state_dict=sd, epoch=ep + 1, val_f1=f1, ema=use_ema, **meta), last_path)
        if f1 > best_f1 or cfg['full_train']:
            best_f1, best_ep, bad = f1, ep + 1, 0
            torch.save(dict(state_dict=sd, epoch=ep + 1, val_f1=f1, ema=use_ema, **meta), best_path)
        elif ep >= cfg['freeze_epochs']:
            bad += 1
            if bad >= cfg['patience'] and not cfg['full_train']:
                print(f'early stop (best ep {best_ep}, F1 {best_f1:.4f})'); break
    json.dump(history, open(f'{out}/{tag}_history.json', 'w'))
    print(f'best epoch {best_ep}  val macro-F1 {best_f1:.4f}')

    # ---- reload best, export probabilities (TTA chosen on val)
    del net, ema; torch.cuda.empty_cache()
    net, ck = load_for_inference(best_path)
    allv = ['id', 'flip', 'p1', 'p2']; Vv = predict_views(net, dl_va, allv, mean=mean, std=std)
    mean_of = lambda V, vs: np.mean([V[v] for v in vs], 0)
    f1s = {n: f1_score(yva, mean_of(Vv, vs).argmax(1), average='macro')
           for n, vs in [('none', ['id']), ('flip', ['id', 'flip']), ('flip+perm', allv)]}
    for n, s in f1s.items(): print(f'val macro-F1  TTA={n:10s} {s:.4f}')
    best_tta = max(['none', 'flip', 'flip+perm'], key=lambda k: f1s[k])
    sample = pd.read_csv(f'{comp}/sample_submission.csv'); test_ids = sample.image_id.tolist()
    dl_te = DataLoader(EvalDS([f'{comp}/test_images/{i}' for i in test_ids], eval_tf, cfg['max_side']), batch_size=64, **kw)
    Vt = predict_views(net, dl_te, allv, mean=mean, std=std)
    pick = lambda V: mean_of(V, {'none': ['id'], 'flip': ['id', 'flip'], 'flip+perm': allv}[best_tta])
    np.savez(f'{out}/{tag}_probs.npz', val_ids=np.array(va.image_id.tolist()), y_val=yva, test_ids=np.array(test_ids),
             classes=np.array(classes), val_probs=pick(Vv), test_probs=pick(Vt), tta=best_tta,
             val_probs_flip=mean_of(Vv, ['id', 'flip']), test_probs_flip=mean_of(Vt, ['id', 'flip']),
             val_probs_perm=mean_of(Vv, allv), test_probs_perm=mean_of(Vt, allv))
    pd.DataFrame({'image_id': test_ids, 'animal_class': [classes[i] for i in pick(Vt).argmax(1)]}) \
      .to_csv(f'{out}/{tag}_submission.csv', index=False)
    print(f'TTA used: {best_tta}. saved {best_path}, {last_path}, {tag}_probs.npz, {tag}_submission.csv')


if __name__ == '__main__':
    main()

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_fscore_support as prfs

COMP = '/kaggle/input/competitions/ai-of-god-5/participant_data'; W = '/kaggle/working'
Z = np.load(f'{W}/dinov3_b_probs.npz', allow_pickle=True)
classes = list(Z['classes']); cmap = {c: i for i, c in enumerate(classes)}
val_ids = [str(i) for i in Z['val_ids']]; test_ids = [str(i) for i in Z['test_ids']]
yv, pv, pt = Z['y_val'], Z['val_probs'], Z['test_probs']
pred_v, pred_t = pv.argmax(1), pt.argmax(1); cv, ct = pv.max(1), pt.max(1)
print(f'val macro-F1 {f1_score(yv, pred_v, average="macro"):.4f}  acc {(pred_v==yv).mean():.4f}  n_val={len(yv)}')

p, r, f, s = prfs(yv, pred_v, labels=range(72), zero_division=0)
tab = pd.DataFrame(dict(cls=classes, prec=p.round(3), rec=r.round(3), f1=f.round(3), support=s)).sort_values('f1')
print('\nWorst 20 classes on val:\n', tab.head(20).to_string(index=False))

cm = confusion_matrix(yv, pred_v, labels=range(72)); np.fill_diagonal(cm, 0)
pairs = sorted([(cm[i, j], classes[i], classes[j]) for i in range(72) for j in range(72) if cm[i, j]], reverse=True)[:20]
print('\nTop confusions:'); [print(f'  {n}x  true {a} -> pred {b}') for n, a, b in pairs]

# How much of the val->LB gap could be sampling noise / class coverage? (macro-F1 over true U predicted classes)
rng = np.random.default_rng(0)
for frac in (1.0, 0.8, 0.6, 0.4):
    out = []
    for _ in range(300):
        keep = rng.choice(72, int(72 * frac), replace=False)
        idx = np.where(np.isin(yv, keep))[0]; idx = rng.choice(idx, min(len(idx), 300), replace=False)
        out.append(f1_score(yv[idx], pred_v[idx], average='macro'))
    print(f'public subset with {int(frac*100)}% of classes, 300 imgs: macro-F1 {np.mean(out):.3f} +/- {np.std(out):.3f}')

In [ ]:

def show(ids, titles, folder, fn, ncol=6, max_side=256):
    n = len(ids)
    if n == 0:
        print("No images to display.")
        return

    nrow = (n + ncol - 1) // ncol

    fig, ax = plt.subplots(
        nrow, ncol,
        figsize=(3 * ncol, 3.2 * nrow),
        squeeze=False
    )
    axs = ax.ravel()

    for a in axs:
        a.axis("off")

    for a, i, t in zip(axs, ids, titles):
        path = f"{COMP}/{folder}/{i}"

        try:
            with Image.open(path) as img:
                im = img.convert("RGB")
                im.thumbnail((max_side, max_side))
                a.imshow(im)

            a.set_title(t, fontsize=8)

        except Exception as e:
            a.set_title(f"ERROR: {i}", fontsize=8)
            print(f"Failed to load {path}: {e}")

    plt.tight_layout()
    plt.savefig(f"{W}/{fn}", dpi=100, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
wrong = np.where(pred_v != yv)[0]
order = wrong[np.argsort(-cv[wrong])][:24]

show(
    [val_ids[i] for i in order],
    [
        f'TRUE {classes[yv[i]]}\nPRED {classes[pred_v[i]]} ({cv[i]:.2f})'
        for i in order
    ],
    'train_images',
    'val_errors.png'
)

In [ ]:
tr = pd.read_csv(f'{COMP}/train.csv')
prior = tr.animal_class.map(cmap).value_counts(normalize=True).reindex(range(72)).fillna(0).values

# (a) predicted-class histogram vs expected. Classes predicted only 1-2 times may become zero-F1 classes
cnt = np.bincount(pred_t, minlength=72); exp = prior * len(pred_t)
d = pd.DataFrame(dict(cls=classes, pred=cnt, expected=exp.round(1), ratio=(cnt / np.maximum(exp, 1e-9)).round(2)))
print('Classes predicted <=2 times on test:', d[d.pred <= 2].cls.tolist())
print('\nMost under-predicted:\n', d.sort_values('ratio').head(10).to_string(index=False))
print('\nMost over-predicted:\n', d.sort_values('ratio').tail(10).to_string(index=False))

# (b) confidence shift, plus the accuracy the test set "should" have if it behaved like val
bins = np.array([0, .3, .5, .7, .85, .95, 1.0001]); bv = np.digitize(cv, bins) - 1; bt = np.digitize(ct, bins) - 1
acc_b = np.array([(pred_v[bv == b] == yv[bv == b]).mean() if (bv == b).any() else (pred_v == yv).mean() for b in range(6)])
print(f'\nmean conf  val-correct {cv[pred_v==yv].mean():.3f} | val-wrong {cv[pred_v!=yv].mean():.3f} | test {ct.mean():.3f}')
print(pd.DataFrame(dict(bin=[f'{bins[b]:.2f}-{bins[b+1]:.2f}' for b in range(6)],
      val_n=[(bv == b).sum() for b in range(6)], val_acc=acc_b.round(3), test_n=[(bt == b).sum() for b in range(6)])).to_string(index=False))
print(f'Confidence-implied test accuracy if calibrated like val: {acc_b[bt].mean():.3f}   (leaderboard ~0.81)')

# (c) agreement with your other models, on val vs test
def load_probs(path):
    R = np.load(path, allow_pickle=True)
    return [str(i) for i in R['val_ids']], R['val_probs'], [str(i) for i in R['test_ids']], R['test_probs']
def align(ids_src, P, ids_tgt):
    pos = {i: k for k, i in enumerate(ids_src)}; return P[[pos[i] for i in ids_tgt]]

REFS = {'ens_v1': f'{W}/ensemble_state_v1.npz', 'eva02': f'{W}/eva02_b_probs.npz'}   # edit to your saved prob files
ref_t = None
for k, pth in REFS.items():
    if not os.path.exists(pth): continue
    try: vi, vp, ti, tp = load_probs(pth)
    except KeyError as e: print(k, 'missing key', e); continue
    rv, rt = align(vi, vp, val_ids).argmax(1), align(ti, tp, test_ids).argmax(1)
    print(f'{k}: agreement with DINOv3  val {(rv==pred_v).mean():.3f}  test {(rt==pred_t).mean():.3f}')
    if ref_t is None: ref_t = rt

if ref_t is not None:
    dis = np.where(pred_t != ref_t)[0]; dis = dis[np.argsort(-ct[dis])][:24]
    show([test_ids[i] for i in dis], [f'DINOv3 {classes[pred_t[i]]} ({ct[i]:.2f})\nREF {classes[ref_t[i]]}' for i in dis],
         'test_images', 'test_disagreements.png')

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import ks_2samp, spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

def stats(path):
    im = Image.open(path); w, h = im.size; fs = os.path.getsize(path)
    rgb = im.convert('RGB').resize((64, 64)); a = np.asarray(rgb, np.float32) / 255
    hsv = np.asarray(rgb.convert('HSV'), np.float32) / 255
    code = (a * 3.999).astype(int); code = code[..., 0] * 16 + code[..., 1] * 4 + code[..., 2]
    c = np.bincount(code.ravel(), minlength=64) / code.size
    g = a.mean(2)
    return dict(w=w, h=h, aspect=w / h, mp=w * h / 1e6, fsize=fs / 1e3, bpp=fs / (w * h),
                r=a[..., 0].mean(), g=a[..., 1].mean(), b=a[..., 2].mean(), sat=hsv[..., 1].mean(), val=hsv[..., 2].mean(),
                ncol=int((c > 0.01).sum()), dom=c.max(), edge=(np.abs(np.diff(g, axis=1)).mean() + np.abs(np.diff(g, axis=0)).mean()) / 2)

items = [('train', i, f'{COMP}/train_images/{i}') for i in tr.image_id] + [('test', i, f'{COMP}/test_images/{i}') for i in test_ids]
with ThreadPoolExecutor(4) as ex: res = list(ex.map(lambda t: stats(t[2]), items))
M = pd.DataFrame(res); M['split'] = [t[0] for t in items]; M['id'] = [t[1] for t in items]
M.loc[(M.split == 'train') & M.id.isin(val_ids), 'split'] = 'val'; M.to_csv(f'{W}/img_stats.csv', index=False)

feats = [c for c in M.columns if c not in ('split', 'id')]
g = lambda s, f: M.loc[M.split == s, f]
print(pd.DataFrame([(f, ks_2samp(g('train', f), g('test', f)).statistic, ks_2samp(g('val', f), g('test', f)).statistic) for f in feats],
      columns=['feature', 'KS train-vs-test', 'KS val-vs-test']).sort_values('KS train-vs-test', ascending=False).round(3).to_string(index=False))

X = M[feats].fillna(0).values; t = (M.split == 'test').values.astype(int)
rf = RandomForestClassifier(300, min_samples_leaf=5, n_jobs=-1, random_state=0)
pa = cross_val_predict(rf, X, t, cv=5, method='predict_proba')[:, 1]
print(f'\nAdversarial AUC (labelled vs test, from simple stats): {roc_auc_score(t, pa):.3f}   (0.5 = indistinguishable)')
print(pd.Series(rf.fit(X, t).feature_importances_, feats).sort_values(ascending=False).head(6).round(3))

# which stats predict val errors?
V = M[M.split == 'val'].set_index('id').loc[val_ids]; wflag = (pred_v != yv)
cor = {f: spearmanr(V[f], wflag)[0] for f in feats}
for f in sorted(cor, key=lambda k: -abs(cor[k]))[:4]:
    q = pd.qcut(V[f], 4, duplicates='drop')
    print(f'\n{f} (rho={cor[f]:.2f}) val error rate by quartile:'); print(pd.Series(wflag, index=V.index).groupby(q.values).mean().round(3).to_string())

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, timm
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
dev = 'cuda'

class Net(nn.Module):
    def __init__(self, cfg, n_classes):
        super().__init__()
        self.backbone = timm.create_model(cfg['model_name'], img_size=cfg['img_size'], pretrained=False,
                                          num_classes=0, drop_path_rate=cfg['drop_path'])
        self.n_last, self.npfx = cfg['n_last'], self.backbone.num_prefix_tokens
        self._feats = []
        for blk in self.backbone.blocks[-self.n_last:]:
            blk.register_forward_hook(self._hook)
        fd = 2 * self.n_last * self.backbone.num_features
        self.norm = nn.LayerNorm(fd)
        self.head = nn.Sequential(nn.Dropout(cfg['dropout']), nn.Linear(fd, cfg['hidden_dim']), nn.GELU(),
                                  nn.Dropout(cfg['dropout']), nn.Linear(cfg['hidden_dim'], n_classes))
        self.proj = nn.Sequential(nn.Linear(fd, cfg['hidden_dim']), nn.GELU(), nn.Linear(cfg['hidden_dim'], cfg['proj_dim']))
    def _hook(self, module, inp, out): self._feats.append(out[0] if isinstance(out, (tuple, list)) else out)
    def features(self, x):
        self._feats.clear(); self.backbone.forward_features(x)
        outs, fs = list(self._feats), []; self._feats.clear()
        for o in outs:
            o = self.backbone.norm(o); fs += [o[:, 0], o[:, self.npfx:].mean(1)]
        return self.norm(torch.cat(fs, 1))
    def forward(self, x): return self.head(self.features(x))

class EvalDS(Dataset):
    def __init__(self, paths, tf, max_side): self.paths, self.tf, self.ms = paths, tf, max_side
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        if self.ms and max(img.size) > self.ms: img.thumbnail((self.ms, self.ms), Image.BILINEAR)
        return self.tf(img)

# ---- load checkpoint (config, classes and normalization are stored inside it)
ck = torch.load(f'{W}/dinov3_b_best.pt', map_location='cpu', weights_only=False)
cfg = ck['cfg']; assert list(ck['classes']) == classes, 'class order mismatch'
net = Net(cfg, len(ck['classes']))
print(net.load_state_dict(ck['state_dict']))          # must print <All keys matched successfully>
net = net.to(dev).eval()
print('best epoch', ck['epoch'], 'val F1', round(ck['val_f1'], 4))

eval_tf = T.Compose([T.Resize((cfg['img_size'], cfg['img_size']), interpolation=T.InterpolationMode.BICUBIC),
                     T.ToTensor(), T.Normalize(ck['mean'], ck['std'])])

@torch.no_grad()
def embed(paths):
    out = []
    for x in DataLoader(EvalDS(paths, eval_tf, cfg['max_side']), batch_size=64, num_workers=4):
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(F.normalize(net.features(x.to(dev)).float(), dim=1).cpu())
    return torch.cat(out)

tr_only = tr[~tr.image_id.isin(val_ids)]; y_tr = tr_only.animal_class.map(cmap).values
E_tr = embed([f'{COMP}/train_images/{i}' for i in tr_only.image_id])
E_va = embed([f'{COMP}/train_images/{i}' for i in val_ids])
E_te = embed([f'{COMP}/test_images/{i}' for i in test_ids])

sv, iv = (E_va @ E_tr.T).max(1); st, it = (E_te @ E_tr.T).max(1)
print(f'1-NN accuracy on val: {(y_tr[iv.numpy()] == yv).mean():.3f}')
print(f'nearest-train cosine sim   val median {sv.median():.3f} p10 {sv.quantile(.1):.3f} | test median {st.median():.3f} p10 {st.quantile(.1):.3f}')
print(f'1-NN vs head agreement     val {(y_tr[iv.numpy()] == pred_v).mean():.3f} | test {(y_tr[it.numpy()] == pred_t).mean():.3f}')

Xp = PCA(64, random_state=0).fit_transform(torch.cat([E_va, E_te]).numpy())
t = np.r_[np.zeros(len(E_va)), np.ones(len(E_te))]
pa = cross_val_predict(LogisticRegression(max_iter=2000), Xp, t, cv=5, method='predict_proba')[:, 1]
print(f'val-vs-test embedding AUC: {roc_auc_score(t, pa):.3f}   (0.5 = same distribution)')

low = np.argsort(st.numpy())[:24]    # test images least similar to anything in train
show([test_ids[i] for i in low], [f'sim {st[i]:.2f}\nPRED {classes[pred_t[i]]} ({ct[i]:.2f})' for i in low],
     'test_images', 'test_unfamiliar.png')